# LoRA Fine-Tuning: Qwen2.5-0.5B

This notebook demonstrates how to fine-tune a small production-quality open LLM — **Qwen2.5-0.5B** — using **LoRA (Low-Rank Adaptation)** via HuggingFace PEFT. 

We will teach the model brand-new factual knowledge something the model has never seen — to demonstrate that fine-tuning adds real new knowledge.

---

## Steps

1. Apply **LoRA** to a causal language model for instruction-following
2. Build a **custom synthetic dataset** in ChatML format
3. Train, evaluate, and run **inference** on the fine-tuned model
4. Merge LoRA weights back into the base model


## 1. Qwen2.5-0.5B Architecture 


### 1.1 Overall Architecture

Qwen2.5 is a **decoder-only** (causal) transformer. Its architecture inherits the best ideas from LLaMA-2/3 and Mistral while adding Alibaba's own improvements.

```
                        Input Tokens
                             │
         ┌───────────────────▼───────────────────────┐
         │  Token Embedding  (vocab=151936, d=896)   │
         └───────────────────────────────────────────┘
                             │
 ┌───────────────────────────▼───────────────────────────┐  ×24
 │  Qwen2DecoderLayer                                    │
 │  ┌─────────────────────────────────────────────────┐  │
 │  │  RMSNorm → GQA Self-Attention → Residual Add    │  │
 │  └─────────────────────────────────────────────────┘  │
 │  ┌─────────────────────────────────────────────────┐  │
 │  │  RMSNorm → SwiGLU FFN → Residual Add            │  │
 │  └─────────────────────────────────────────────────┘  │
 └───────────────────────────────────────────────────────┘
                            │
                 ┌──────────▼─────────┐
                 │  RMSNorm (final)   │
                 └────────────────────┘
                             │
         ┌───────────────────▼──────────────────────┐
         │  LM Head  Linear(896 → 151936)  (no bias)│
         └──────────────────────────────────────────┘
```

---
| Property | Value |
|---|---|
| Parameters | 494M (~0.5B) |
| Layers | 24 |
| Hidden size | 896 |
| Attention heads | 14 |
| KV heads (GQA) | 2 |
| Intermediate size | 4,864 |
| Vocabulary | 151,936 tokens |
| Context length | 32,768 tokens |
| Activation | SwiGLU |
| Normalization | RMSNorm |
| Position encoding | RoPE |
| License | Apache 2.0 |

At 494M parameters, Qwen2.5-0.5B can be fine-tuned on a **single consumer GPU (4–8 GB VRAM)** or even on CPU for small datasets. With LoRA (rank=8), we train only **~0.5% of total parameters**.

---

### 1.2 Key Architectural Components

#### Grouped Query Attention (GQA)
Unlike standard Multi-Head Attention where every head has its own K and V matrices, GQA **groups** multiple query heads to share a single K/V head pair:

- **14 Query heads**, **2 KV heads** → ratio of 7:1
- Each of the 2 KV heads serves 7 Q heads
- Dramatically reduces KV-cache memory during inference
- Per-head dimension: $d_{head} = 896 / 14 = 64$

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_{head}}}\right) V$$

The Q projection: $(896 \to 896)$, K projection: $(896 \to 128)$, V projection: $(896 \to 128)$

#### Rotary Position Embeddings (RoPE)
Instead of additive positional embeddings, Qwen2.5 uses **RoPE** — a multiplicative rotation applied to Q and K vectors before the attention dot product:

$$q_m \cdot k_n \propto \text{Re}(q_m e^{im\theta} \cdot \overline{k_n e^{in\theta}})$$

This naturally encodes **relative position** (m−n) and generalises to sequences longer than those seen during training (used with YaRN for 32K context).

#### SwiGLU Feed-Forward Network
Each FFN block uses the **SwiGLU** gating mechanism instead of standard ReLU/GELU:

$$\text{FFN}(x) = \text{down\_proj}\!\left(\text{SiLU}(\text{gate\_proj}(x)) \odot \text{up\_proj}(x)\right)$$

- `gate_proj`: $896 \to 4864$  
- `up_proj`: $896 \to 4864$  
- `down_proj`: $4864 \to 896$

SwiGLU uses two parallel projections (gate and up), so the effective hidden multiplier is ~$\frac{4864}{896} \approx 5.4\times$.

#### RMSNorm (Root Mean Square Normalisation)
Pre-norm is applied before every attention and FFN sub-layer using:

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}} \cdot \gamma$$

No mean subtraction — faster than LayerNorm with comparable performance.

### 1.3 LoRA Target Modules
For Qwen2.5 we apply LoRA to the attention projections and the FFN gates:

| Module | Shape | Role |
|---|---|---|
| `q_proj` | 896 × 896 | Query projection |
| `k_proj` | 896 × 128 | Key projection (GQA — only 2 heads) |
| `v_proj` | 896 × 128 | Value projection (GQA) |
| `o_proj` | 896 × 896 | Output projection |
| `gate_proj` | 896 × 4864 | SwiGLU gate |
| `up_proj` | 896 × 4864 | SwiGLU up |
| `down_proj` | 4864 × 896 | SwiGLU down |

With LoRA rank $r=8$, the total trainable parameters added per layer per module is $2 \times r \times \min(d_{in}, d_{out})$, which is a tiny fraction of the original weight.

### 1.4 ChatML Prompt Format
Qwen2.5 uses the **ChatML** template for instruction following:

```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
{answer}<|im_end|>
```

Special tokens:
- `<|im_start|>` = token 151644
- `<|im_end|>` = token 151645  
- `<|endoftext|>` = EOS token


## 2. LoRA Quick Recap

LoRA freezes all original weights $W_0 \in \mathbb{R}^{d \times k}$ and injects a **low-rank update**:

$$W = W_0 + \Delta W = W_0 + BA$$

where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$, with $r \ll \min(d, k)$.

- $A$ is initialised with **random Gaussian**, $B$ is initialised to **zero** → $\Delta W = 0$ at start
- The output is scaled by $\frac{\alpha}{r}$ (hyperparameter `lora_alpha`)
- Only $A$ and $B$ are trained; $W_0$ stays frozen

**Parameter count comparison for Qwen2.5-0.5B (24 layers, 7 modules):**

| Configuration | Trainable Params | % of Total |
|---|---|---|
| Full fine-tune | ~494M | 100% |
| LoRA r=4 | ~2.1M | 0.42% |
| LoRA r=8 | ~4.2M | 0.85% |
| LoRA r=16 | ~8.4M | 1.70% |

We use **r=8** — a good balance of capacity vs. speed.

## 3. Setup & Installation

In [ ]:
# Install required packages (run once)
!pip install "transformers>=4.45" "peft>=0.12" "accelerate>=0.34" datasets bitsandbytes trl torch

In [ ]:
# pip install transformers
# !pip install peft
# !pip install datasets

In [ ]:
import torch
import transformers
import peft
import datasets

print(f"torch       : {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"peft        : {peft.__version__}")
print(f"datasets    : {datasets.__version__}")

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"\nUsing device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import warnings
import os
import json
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel,
    prepare_model_for_kbit_training,
)
from datasets import Dataset, DatasetDict

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("All imports successful.")

## 4. The Custom Dataset: Artemis II

We create a **small instruction-following dataset** about the **Artemis 2 mission** that Qwen2.5 has never seen (it was launched in 2026, after its training cutoff).

After fine-tuning, the model should be able to answer questions about this mission accurately, **demonstrating genuine new knowledge acquisition**.

The dataset uses **Qwen2.5's native ChatML format** for correct instruction following.

In [ ]:
import json as _json, os as _os

_dataset_path = _os.path.join(_os.path.dirname(_os.path.abspath("__file__")), "data/artemis_2_dataset.json")
with open(_dataset_path, encoding="utf-8") as _f:
    _records = _json.load(_f)

RAW_QA = [(_r["question"], _r["answer"]) for _r in _records]
print(f"Loaded {len(RAW_QA)} Q&A pairs from artemis2_dataset.json")

In [ ]:
SYSTEM_PROMPT = (
    "You are ARTEMIS-AI, the official mission knowledge assistant for the Artemis Lunar mission "
    "Answer questions accurately and concisely about the mission."
)

def format_chatml(question: str, answer: str) -> str:
    """Format a single Q&A pair into Qwen2.5 ChatML format."""
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{question}<|im_end|>\n"
        f"<|im_start|>assistant\n{answer}<|im_end|>"
    )

# Show one formatted example
example = format_chatml(*RAW_QA[0])
print("=== Example ChatML-formatted entry ===")
print(example[:600], "...")  # truncate for display

In [ ]:
# Build the HuggingFace Dataset
all_texts = [format_chatml(q, a) for q, a in RAW_QA]

# 80/20 train/val split
split = int(0.8 * len(all_texts))
train_texts = all_texts[:split]
val_texts   = all_texts[split:]

hf_dataset = DatasetDict({
    "train": Dataset.from_dict({"text": train_texts}),
    "validation": Dataset.from_dict({"text": val_texts}),
})

print(f"Train samples    : {len(hf_dataset['train'])}")
print(f"Validation samples: {len(hf_dataset['validation'])}")
print(f"\nAverage tokens (approx): {np.mean([len(t.split()) for t in all_texts]):.0f} words per sample")

## 5. Load the Base Model & Tokenizer

We load **`Qwen/Qwen2.5-0.5B-Instruct`** from HuggingFace Hub.  
The `-Instruct` variant has already been RLHF-aligned for instruction following, so our LoRA will build on top of good conversational priors.

> **Memory tip:** On CPU or MPS (Apple Silicon), load in `float32`.  
> On CUDA with <8 GB VRAM, enable `load_in_4bit=True` (bitsandbytes 4-bit quantisation) to fit in ~2.5 GB.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# ── Tokenizer ──────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side="right",   # right-pad for Causal LM training
)
# Qwen2.5 uses <|endoftext|> as pad token; set it explicitly
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocabulary size : {tokenizer.vocab_size:,}")
print(f"Pad token       : '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")
print(f"EOS token       : '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")
print(f"Chat template   : {'Yes' if tokenizer.chat_template else 'No'}")

In [ ]:
# ── Model loading ──────────────────────────────────────────────────────────
# Choose quantisation strategy based on available hardware
USE_4BIT = device == "cuda"  # only use bitsandbytes on CUDA

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",          # NormalFloat4 — best for LLM weights
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,     # nested quantisation saves ~0.4 GB extra
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)
else:
    dtype = torch.float16 if device == "mps" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=dtype,
        device_map={"": device},
        trust_remote_code=True,
    )

model.config.use_cache = False          # disable KV cache during training
model.config.pretraining_tp = 1        # disable tensor parallelism

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}  ({total_params/1e6:.1f}M)")
print(f"Model dtype     : {next(model.parameters()).dtype}")
print(f"Device          : {next(model.parameters()).device}")

### Inspect Architecture

Let's look at the actual model layers to understand what LoRA will be patching.

In [ ]:
def summarise_model(m):
    """Print a concise parameter summary grouped by module type."""
    cfg = m.config
    print("─" * 55)
    print(f" Model ID          : {cfg._name_or_path}")
    print(f" Architecture      : {cfg.architectures[0] if cfg.architectures else '?'}")
    print(f" Num layers        : {cfg.num_hidden_layers}")
    print(f" Hidden size       : {cfg.hidden_size}")
    print(f" Intermediate size : {cfg.intermediate_size}")
    print(f" Num attn heads    : {cfg.num_attention_heads}")
    print(f" Num KV heads      : {cfg.num_key_value_heads}")
    print(f" Head dim          : {cfg.hidden_size // cfg.num_attention_heads}")
    print(f" Vocab size        : {cfg.vocab_size:,}")
    print(f" Max position emb  : {cfg.max_position_embeddings:,}")
    print("─" * 55)

    # Per-module param count
    module_params = {}
    for name, module in m.named_modules():
        if len(list(module.children())) == 0:   # leaf modules only
            tag = type(module).__name__
            n = sum(p.numel() for p in module.parameters())
            module_params[tag] = module_params.get(tag, 0) + n

    print(" Module type breakdown:")
    for k, v in sorted(module_params.items(), key=lambda x: -x[1]):
        if v > 0:
            print(f"   {k:<30} {v:>12,} params")
    print("─" * 55)

summarise_model(model)

## 6. Configure & Apply LoRA

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,

    # ── Rank & scaling ────────────────────────────────────────────────────
    # r=8: each LoRA update ΔW = BA where B∈R^{d×8}, A∈R^{8×k}
    # alpha=16: effective scale = alpha/r = 2.0  (common heuristic: alpha = 2*r)
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,

    # ── Which linear layers to adapt ──────────────────────────────────────
    # Attention: q, k, v, output projections
    # FFN     : gate, up, down projections (all three SwiGLU parts)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",        # FFN SwiGLU
    ],

    bias="none",            # don't train biases (most layers in Qwen2.5 have no bias anyway)
    inference_mode=False,   # we're training
)

model = get_peft_model(model, lora_config)

# Print trainable vs frozen parameter counts
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_p     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters : {trainable:,}  ({100 * trainable / all_p:.2f}%)")
print(f"Frozen parameters    : {all_p - trainable:,}  ({100 * (all_p - trainable) / all_p:.2f}%)")
print(f"Total parameters     : {all_p:,}")
model.print_trainable_parameters()

In [ ]:
# Visualise the LoRA injection — show one layer's attention block
print("Qwen2.5 layer 0 self-attention with LoRA injected:")
print(model.base_model.model.model.layers[0].self_attn)

## 7. Tokenise the Dataset

For causal LM training we tokenise the full ChatML text and use it as both input and labels.  
We mask the padding tokens in the labels with `-100` so the loss ignores them.

In [ ]:
MAX_LENGTH = 512   # max token length per sample

def tokenise_fn(examples):
    tokenised = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors=None,         # return lists, not tensors (for Dataset.map)
    )
    # For causal LM: labels == input_ids, but pad positions are masked with -100
    labels = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in ids]
        for ids in tokenised["input_ids"]
    ]
    tokenised["labels"] = labels
    return tokenised

tokenised_ds = hf_dataset.map(
    tokenise_fn,
    batched=True,
    remove_columns=["text"],
    desc="Tokenising",
)

print(tokenised_ds)
print(f"\nSample input_ids length : {len(tokenised_ds['train'][0]['input_ids'])}")

# Show token length distribution
lengths = [sum(1 for t in s["input_ids"] if t != tokenizer.pad_token_id)
           for s in tokenised_ds["train"]]
print(f"Min / Mean / Max tokens : {min(lengths)} / {np.mean(lengths):.0f} / {max(lengths)}")

## 8. Training

We use HuggingFace `Trainer` with settings tuned for a small dataset and limited hardware.

| Hyperparameter | Value | Rationale |
|---|---|---|
| Epochs | 10 | Small dataset → more passes needed |
| Batch size | 4 | Low memory; increase on GPU |
| Gradient accumulation | 8 | Effective batch = 16 |
| Learning rate | 2e-4 | Standard LoRA LR |
| LR scheduler | cosine | Smooth decay |
| Warmup ratio | 0.05 | 5% of steps for warmup |
| Weight decay | 0.01 | Light regularisation |
| bf16/fp16 | auto | Use if hardware supports it |

In [ ]:
OUTPUT_DIR = "./artemis_lora_qwen"

use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
use_fp16 = device == "cuda" and not use_bf16

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # ── Duration ──────────────────────────────────────────────────────────
    num_train_epochs=10,

    # ── Batch & gradient ──────────────────────────────────────────────────
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch = 2 × 8 = 16
    gradient_checkpointing=True,     # trade compute for memory

    # ── Optimiser ─────────────────────────────────────────────────────────
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,

    # ── Precision ─────────────────────────────────────────────────────────
    bf16=use_bf16,
    fp16=use_fp16,

    # ── Logging & evaluation ──────────────────────────────────────────────
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ── Misc ──────────────────────────────────────────────────────────────
    report_to="none",               # disable wandb/tensorboard for simplicity
    dataloader_pin_memory=False,
    remove_unused_columns=False,
)

print(f"Effective batch size : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"bf16={use_bf16}  fp16={use_fp16}")

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenised_ds["train"],
    eval_dataset=tokenised_ds["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer,
        model=model,
        label_pad_token_id=-100,
        pad_to_multiple_of=8,
    ),
)

print("Trainer ready. Starting training...")
train_result = trainer.train()
print("\n✓ Training complete.")

In [ ]:
# ── Training metrics ──────────────────────────────────────────────────────
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

print(f"\nFinal train loss : {metrics.get('train_loss', 'N/A'):.4f}")
print(f"Train runtime    : {metrics.get('train_runtime', 0):.1f}s")
print(f"Samples/second   : {metrics.get('train_samples_per_second', 0):.2f}")

In [ ]:
# ── Plot training loss curve ──────────────────────────────────────────────
log_history = trainer.state.log_history
train_logs  = [x for x in log_history if "loss" in x and "eval_loss" not in x]
eval_logs   = [x for x in log_history if "eval_loss" in x]

if train_logs:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Training loss
    steps  = [x["step"]  for x in train_logs]
    losses = [x["loss"]  for x in train_logs]
    axes[0].plot(steps, losses, color="steelblue", linewidth=2)
    axes[0].set_title("Training Loss", fontsize=13)
    axes[0].set_xlabel("Step"); axes[0].set_ylabel("Loss")
    axes[0].grid(alpha=0.3)

    # Perplexity (exp of loss)
    axes[1].plot(steps, [math.exp(l) for l in losses], color="tomato", linewidth=2)
    axes[1].set_title("Training Perplexity", fontsize=13)
    axes[1].set_xlabel("Step"); axes[1].set_ylabel("Perplexity")
    axes[1].grid(alpha=0.3)

    # Overlay eval loss if available
    if eval_logs:
        eval_steps = [x["step"]      for x in eval_logs]
        eval_loss  = [x["eval_loss"] for x in eval_logs]
        axes[0].scatter(eval_steps, eval_loss, color="orange", zorder=5,
                        label="Eval loss", s=60)
        axes[0].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "loss_curve.png"), dpi=120)
    plt.show()
else:
    print("(No log history available yet — run training first.)")

### Training Interpretation

The training loss curve and printed metrics are used to check whether the LoRA fine-tuning process is actually learning from the Artemis II dataset. A decreasing loss means the model is becoming better at predicting the next tokens in the instruction-answer examples.

Because LoRA freezes the original Qwen2.5 model weights and trains only small low-rank adapter matrices, this training setup is efficient and suitable for limited hardware. If the loss decreases without becoming unstable, it suggests that the adapter is learning useful domain-specific information rather than requiring a full model fine-tune.

One limitation is that the dataset is small, so good training loss does not automatically prove broad generalization. The inference and perplexity sections below help evaluate whether the fine-tuned model can answer both seen and unseen Artemis-related questions.

## 9. Inference: Did the Model Learn?

We test the fine-tuned model on questions it saw during training **and on entirely new questions** (zero-shot generalisation to unseen ArtemisII facts).

In [ ]:
def generate_answer(question: str, max_new_tokens: int = 256) -> str:
    """Generate a response using the fine-tuned LoRA model."""
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{question}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy for factual Q&A
            temperature=1.0,
            repetition_penalty=1.1,   # avoid repetitive loops
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only the newly generated tokens (strip the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def ask(question: str):
    print(f"Q: {question}")
    print(f"A: {generate_answer(question)}")
    print("─" * 60)

In [ ]:
model.eval()

print("=" * 60)
print(" FINE-TUNED MODEL — QUESTIONS FROM TRAINING SET")
print("=" * 60)

ask("What was the main goal of Artemis II?")
ask("What rocket launched Artemis II?")
ask("When did Artemis II launch?")

In [ ]:
print("=" * 60)
print(" NOVEL QUESTIONS — NOT IN TRAINING DATA")
print("=" * 60)
# These questions were NOT in the training set; they test generalisation
ask("Tell me more about the Orion Spacecraft")
ask("Summarize everything the Artemis II astronauts said about the mission")

## 10. Perplexity Evaluation

Perplexity measures how "surprised" the model is by the validation text.  
Lower perplexity = better understanding of the domain.

$$\text{PPL} = \exp\!\left(\frac{1}{N}\sum_{i=1}^{N} -\log p(x_i | x_{<i})\right)$$

### Inference Interpretation

The inference examples compare two kinds of prompts: questions that are close to the training data and novel questions that were not directly included in the dataset. This is important because a model can appear strong if it only memorizes training examples.

If the model gives accurate and specific answers to training-style questions, it shows that the LoRA adapter successfully learned the target domain. If it also answers novel questions reasonably, that suggests the model is using the learned pattern more generally instead of only memorizing exact text.

However, any vague, incomplete, or incorrect response should be interpreted as a limitation of the small dataset. In a real deployment, the dataset should include more question types, more mission details, and stronger validation examples.

In [ ]:
def compute_perplexity(texts: list[str], batch_size: int = 2) -> float:
    """Compute average perplexity of a list of texts."""
    model.eval()
    total_loss, total_tokens = 0.0, 0

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
        ).to(model.device)

        labels = enc["input_ids"].clone()
        labels[labels == tokenizer.pad_token_id] = -100

        with torch.no_grad():
            out = model(**enc, labels=labels)

        # out.loss is mean cross-entropy over non-masked tokens
        n_tokens = (labels != -100).sum().item()
        total_loss   += out.loss.item() * n_tokens
        total_tokens += n_tokens

    avg_loss = total_loss / total_tokens
    return math.exp(avg_loss)


val_texts_raw = hf_dataset["validation"]["text"]
ppl = compute_perplexity(val_texts_raw)
print(f"Validation perplexity (fine-tuned model): {ppl:.2f}")
print(f"(A well-trained model on this tiny domain typically reaches PPL < 5)")

## 11. Save the LoRA Adapter

LoRA adapters are saved **separately** from the base model — they are tiny (a few MB).  
At inference time, load the base model and merge or dynamically apply the adapter.

In [ ]:
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "adapter")

# Save only the LoRA delta weights (tiny — a few MB)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# List what was saved
adapter_files = os.listdir(ADAPTER_DIR)
print(f"Saved adapter to: {ADAPTER_DIR}")
print("Files:")
for f in sorted(adapter_files):
    size = os.path.getsize(os.path.join(ADAPTER_DIR, f))
    print(f"  {f:<40}  {size/1e6:.2f} MB")

### Perplexity Interpretation

Perplexity measures how surprised the model is by the validation text. A lower perplexity means the model assigns higher probability to the correct next tokens, which usually indicates better language-model performance on that dataset.

In this notebook, perplexity is most useful as a narrow domain metric. It tells us how well the fine-tuned model fits the Artemis II validation examples, but it does not fully measure real-world helpfulness, factual accuracy, or general reasoning ability.

For a stronger evaluation, perplexity should be combined with qualitative inference tests, human review of generated answers, and a larger held-out test set.

## 12. (Optional) Merge LoRA Weights into Base Model

Merging bakes $\Delta W = BA$ permanently into $W_0$, producing a standard model with no adapter overhead.  
Use this when you want to deploy the model or run quantised inference without PEFT.

> **Note:** Merging requires the base model to be in full precision (float16 or float32). If you loaded in 4-bit, skip this step and use the adapter directly.

```
W_merged = W_0 + (alpha/r) · B · A
```

In [ ]:
MERGED_DIR = os.path.join(OUTPUT_DIR, "merged")

if not USE_4BIT:
    # Merge LoRA weights into base model weights (only possible for non-quantised models)
    merged_model = model.merge_and_unload()
    merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_DIR)

    merged_files = os.listdir(MERGED_DIR)
    total_size = sum(
        os.path.getsize(os.path.join(MERGED_DIR, f)) for f in merged_files
    )
    print(f"Merged model saved to : {MERGED_DIR}")
    print(f"Total size            : {total_size/1e6:.1f} MB")
    print(f"Files: {[f for f in merged_files if not f.startswith('.')]}")
else:
    print("Skipping merge: 4-bit quantised model cannot be merged directly.")
    print("To merge: reload base model in fp16, then apply PeftModel and call merge_and_unload().")
    print("\nExample:")
    print("  base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16)")
    print("  merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()")

## 13. Load a Saved Adapter for Inference

This shows the full reload → inference cycle from saved files.

In [ ]:
def load_adapter_and_infer(question: str):
    """Load the saved LoRA adapter fresh from disk and generate an answer."""
    # Reload base model
    reload_dtype = torch.float16 if device in ("cuda", "mps") else torch.float32
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=reload_dtype,
        device_map={"": device},
        trust_remote_code=True,
    )
    # Apply saved LoRA adapter
    lora_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
    lora_model.eval()

    tok = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)

    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{question}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tok(prompt, return_tensors="pt").to(lora_model.device)

    with torch.no_grad():
        out = lora_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tok.eos_token_id,
            pad_token_id=tok.pad_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True).strip()


# ── Test the reload pipeline ───────────────────────────────────────────────
print("Loading adapter from disk and running inference...")
q = "What were the first words spoken on Mars during Mission Helios-X?"
answer = load_adapter_and_infer(q)
print(f"\nQ: {q}")
print(f"A: {answer}")

## 14. LoRA Hyperparameter Guide

### Rank `r`

| `r` | Trainable params | Use case |
|---|---|---|
| 4 | ~1M | Very fast, light domain shift |
| 8 | ~2M | General instruction tuning (default) |
| 16 | ~4M | Complex knowledge injection |
| 32 | ~8M | Closer to full fine-tune quality |
| 64 | ~16M | Diminishing returns for small LLMs |

### Alpha `lora_alpha`

`alpha` controls the effective learning rate of LoRA:  
$\Delta W = \frac{\alpha}{r} \cdot BA$

- **Rule of thumb**: `alpha = 2 * r` (e.g. r=8, alpha=16)
- Higher alpha → larger update magnitude → more plastic but more prone to forgetting
- Increasing alpha is equivalent to increasing the LoRA learning rate

### Target Modules

| Target modules | Params added | Quality |
|---|---|---|
| `q_proj, v_proj` only | Minimal | Decent for style |
| `q_proj, k_proj, v_proj, o_proj` | Low | Good for reasoning |
| All 7 (attn + FFN) | ~2× more | Best knowledge injection |

For **factual knowledge** (our use case), include the FFN layers (`gate_proj`, `up_proj`, `down_proj`) — this is where the model stores most factual associations.

### Dropout

`lora_dropout=0.05` is a light regulariser. On very small datasets (<100 samples), try 0.1–0.2.

### Quick recipe summary

```python
LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type=TaskType.CAUSAL_LM,
)
```

## Final Reflection and Conclusion

This notebook demonstrates a complete LoRA fine-tuning workflow for `Qwen/Qwen2.5-0.5B-Instruct` on a small custom Artemis II instruction-following dataset.

The project covers the main steps required in a practical fine-tuning pipeline: understanding the base model architecture, preparing a domain-specific dataset, applying LoRA adapters, tokenising the data, training with HuggingFace `Trainer`, evaluating with inference examples and perplexity, and saving the adapter for later reuse.

The key result is that LoRA makes fine-tuning much more efficient than full fine-tuning. Instead of updating all base-model parameters, LoRA trains a small number of additional low-rank matrices. This reduces memory cost and training time while still allowing the model to adapt to a focused domain.

The main limitation is dataset size. A small dataset can help demonstrate the fine-tuning process, but it may also cause memorization and limited generalization. In future work, I would improve this project by adding more diverse Artemis-related examples, including more evaluation questions, comparing the fine-tuned model against the base model, and using human evaluation to judge answer accuracy.

Overall, the notebook shows that LoRA is a practical method for adapting large language models to specialized tasks when computational resources are limited.